
# DL Assignment 03

**Name: Md.Atikur Rahaman**

**Course Email: atikurrahaman0305@gmail.com**  


## End of Assignment

Before submitting:
- Run all cells from top to bottom.  
- Check that all answer sections are filled.  
- Instruction video অনুযায়ী আমাদের দেয়া Colab ফাইলটি থেকে প্রথম একটি Save copy in drive করে নিবা। এরপর Google colab এর মধ্যে কোডগুলো করবে এবং সেই ফাইলটি ‘Anyone with the link’ & ‘View’ Access দিয়ে ফাইলটির Shareble Link টি সাবমিট করবে।

# General Instruction

You must choose your own dataset.

The dataset must:

Be a supervised learning dataset (Regression or Binary Classification)

Contain at least 300 samples

Have at least 2 input features

Be in CSV format

You are NOT allowed to use Dataset or DataLoader.

You must implement everything manually.

# Question 01: [ Marks 05 ]

## Dataset Preparation

### Explanation (Simple)
- Load `insurance.csv`.
- Check missing values.
- Encode categorical columns (`sex`, `smoker`, `region`).
- Split into `X` and `y` (`charges`).
- Train-test split: 80:20.
- Scale `X` features.
- Convert to NumPy arrays and PyTorch tensors.

In [13]:
# Write Answer 01
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load
df = pd.read_csv('insurance.csv')
print(df.shape)
display(df.head())
print(df.isnull().sum())

# Encode
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

# Split X, y
X = df.drop('charges', axis=1)
y = df['charges']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Numpy
X_train_np = np.array(X_train, dtype=np.float32)
X_test_np = np.array(X_test, dtype=np.float32)
y_train_np = np.array(y_train, dtype=np.float32).reshape(-1, 1)
y_test_np = np.array(y_test, dtype=np.float32).reshape(-1, 1)

# Tensor
X_train_tensor = torch.from_numpy(X_train_np)
X_test_tensor = torch.from_numpy(X_test_np)
y_train_tensor = torch.from_numpy(y_train_np)
y_test_tensor = torch.from_numpy(y_test_np)

print(X_train_tensor.shape, X_test_tensor.shape)
print(y_train_tensor.shape, y_test_tensor.shape)


(1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
torch.Size([1070, 8]) torch.Size([268, 8])
torch.Size([1070, 1]) torch.Size([268, 1])


# Question 02: [ Marks 20 ]

## Design a neural network using nn.Module.

### Explanation (Simple)
- Input -> hidden -> output architecture.
- Hidden neurons = 16.
- Activation = `ReLU`.
- Output neuron = 1 for regression.
- Print total trainable parameters.

### The model must contain:

Input layer

At least one hidden layer

Output layer

Suitable activation function



## Justify:

Number of hidden neurons

Choice of activation function

Print  the total number of trainable parameters.


In [14]:
# Write Answer 02
import torch.nn as nn

class InsuranceRegressor(nn.Module):
    def __init__(self, num_features, hidden_units=16):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, 1)
        )

    def forward(self, x):
        return self.network(x)

input_dim = X_train_tensor.shape[1]
model = InsuranceRegressor(input_dim, hidden_units=16)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Total trainable parameters:', total_params)


InsuranceRegressor(
  (network): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)
Total trainable parameters: 161


# Question 03: [ Marks 10 ]

### Explanation (Simple)
- `charges` is continuous, so this is regression.
- Loss function: `MSELoss`.
- Optimizer: `Adam` with small learning rate for stable training.

In [15]:
# Write Answer 03
loss_function = nn.MSELoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(loss_function)
print(optimizer)


MSELoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


# Question 04: [ Marks 15 ]

### Explanation (Simple)
- Training steps: forward -> loss -> zero grad -> backward -> step.
- Epochs = 200.
- Print loss every 10 epochs.
- Save loss in `train_loss_history`.

In [16]:
# Write Answer 04
epochs = 200
train_loss_history = []

for epoch in range(epochs):
    # Forward
    y_pred = model(X_train_tensor)

    # Loss
    loss = loss_function(y_pred, y_train_tensor)

    # Zero grad
    optimizer.zero_grad()

    # Backward
    loss.backward()

    # Update
    optimizer.step()

    train_loss_history.append(loss.item())

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss = {loss.item():.4f}')


Epoch 10, Loss = 322445216.0000
Epoch 20, Loss = 322442848.0000
Epoch 30, Loss = 322440352.0000
Epoch 40, Loss = 322437728.0000
Epoch 50, Loss = 322434912.0000
Epoch 60, Loss = 322431872.0000
Epoch 70, Loss = 322428704.0000
Epoch 80, Loss = 322425184.0000
Epoch 90, Loss = 322421408.0000
Epoch 100, Loss = 322417344.0000
Epoch 110, Loss = 322412928.0000
Epoch 120, Loss = 322408128.0000
Epoch 130, Loss = 322402880.0000
Epoch 140, Loss = 322397376.0000
Epoch 150, Loss = 322391328.0000
Epoch 160, Loss = 322384928.0000
Epoch 170, Loss = 322378080.0000
Epoch 180, Loss = 322370752.0000
Epoch 190, Loss = 322362944.0000
Epoch 200, Loss = 322354688.0000


# Question 05: [ Marks 10 ]

### Explanation (Simple)
- Predict on train and test data.
- Report `MSE` and `MAE`.
- Compare train vs test performance.
- Decide underfitting/overfitting/generalization.

In [17]:
# Write Answer 05
from sklearn.metrics import mean_squared_error, mean_absolute_error

model.eval()
with torch.no_grad():
    y_train_pred = model(X_train_tensor)
    y_test_pred = model(X_test_tensor)

mse_train = mean_squared_error(y_train_np, y_train_pred.numpy())
mae_train = mean_absolute_error(y_train_np, y_train_pred.numpy())
mse_test = mean_squared_error(y_test_np, y_test_pred.numpy())
mae_test = mean_absolute_error(y_test_np, y_test_pred.numpy())

print('Train MSE:', round(mse_train, 4))
print('Train MAE:', round(mae_train, 4))
print('Test MSE :', round(mse_test, 4))
print('Test MAE :', round(mae_test, 4))

if mse_train < 0.7 * mse_test:
    fit_status = 'Overfitting'
elif mse_train > 1.3 * mse_test:
    fit_status = 'Underfitting'
else:
    fit_status = 'Good generalization'

print('Model status:', fit_status)


Train MSE: 322353820.0
Train MAE: 13343.01
Test MSE : 323329440.0
Test MAE : 12965.281
Model status: Good generalization


# Question 06: [ Marks 20 ]

### Explanation (Simple)
- Modify one hyperparameter.
- Here: hidden neurons 16 -> 32.
- Train again.
- Compare final MSE and MAE.


In [18]:
# Write Answer 06
# Change: hidden units 16 -> 32
model_modified = InsuranceRegressor(input_dim, hidden_units=32)
loss_modified = nn.MSELoss()
optimizer_modified = torch.optim.Adam(model_modified.parameters(), lr=0.001)

epochs_modified = 200
modified_loss_history = []

for epoch in range(epochs_modified):
    y_pred_mod = model_modified(X_train_tensor)
    l_mod = loss_modified(y_pred_mod, y_train_tensor)

    optimizer_modified.zero_grad()
    l_mod.backward()
    optimizer_modified.step()

    modified_loss_history.append(l_mod.item())

    if (epoch + 1) % 10 == 0:
        print(f'[Modified] Epoch {epoch+1}, Loss = {l_mod.item():.4f}')

model_modified.eval()
with torch.no_grad():
    y_test_pred_mod = model_modified(X_test_tensor)

mse_test_mod = mean_squared_error(y_test_np, y_test_pred_mod.numpy())
mae_test_mod = mean_absolute_error(y_test_np, y_test_pred_mod.numpy())

print('Original Test MSE:', round(mse_test, 4))
print('Original Test MAE:', round(mae_test, 4))
print('Modified Test MSE:', round(mse_test_mod, 4))
print('Modified Test MAE:', round(mae_test_mod, 4))


[Modified] Epoch 10, Loss = 322445952.0000
[Modified] Epoch 20, Loss = 322442080.0000
[Modified] Epoch 30, Loss = 322438112.0000
[Modified] Epoch 40, Loss = 322433920.0000
[Modified] Epoch 50, Loss = 322429504.0000
[Modified] Epoch 60, Loss = 322424640.0000
[Modified] Epoch 70, Loss = 322419296.0000
[Modified] Epoch 80, Loss = 322413472.0000
[Modified] Epoch 90, Loss = 322407104.0000
[Modified] Epoch 100, Loss = 322400160.0000
[Modified] Epoch 110, Loss = 322392544.0000
[Modified] Epoch 120, Loss = 322384192.0000
[Modified] Epoch 130, Loss = 322375232.0000
[Modified] Epoch 140, Loss = 322365440.0000
[Modified] Epoch 150, Loss = 322354880.0000
[Modified] Epoch 160, Loss = 322343488.0000
[Modified] Epoch 170, Loss = 322331264.0000
[Modified] Epoch 180, Loss = 322318208.0000
[Modified] Epoch 190, Loss = 322304192.0000
[Modified] Epoch 200, Loss = 322289216.0000
Original Test MSE: 323329440.0
Original Test MAE: 12965.281
Modified Test MSE: 323254300.0
Modified Test MAE: 12963.324


# Question 07: [ Marks 20 ]

## Training Analysis (Simple)
- Reset gradients each epoch because gradients accumulate in PyTorch.
- If learning rate is too high, loss becomes unstable.
- If learning rate is too low, training becomes very slow.
- Define layers in `__init__` so parameters are registered once and updated correctly.


In [19]:
# Write Answer 07
# Theory explanation is in the markdown question cell above.
